# Notebook 26 — Attribution conventions, model identity and training recipes

**Author:** Sunita Sahu · **Revision items:** A8 (identity, Molmo connector), A7 (ViLT attribution), A3 (trainable shares)

| Section | Reviewer item | Question |
|---|---|---|
| 1 | A7, A8 | Does every model use the same rule for what counts as vision, fusion, language and head? |
| 2 | A8 | Which repository and commit was measured, base or instruct, and has it changed since? |
| 3 | A3 | Which parameters did each model update in its final multimodal training stage? |

Nothing here needs weights or a GPU. Results go to new `nb26_*` files; the per-model JSON records keep the A13 attribution until the revised conventions are approved (see `paper/Revision_Decision_Log.md`).

---
## Setup

In [1]:
import os, sys, json
from pathlib import Path
import pandas as pd

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
NOTEBOOK_DIR = Path.cwd()
CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CODE_ACTIVE))
MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
ARCHIVE_DIR = CODE_ACTIVE / '_archive' / 'pre_A13_measurements'

from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
from vlm_audit.parameter_layouts import ROSTER_TABLE_NAMES, CONFIG_MIRRORS, load_parameter_shapes
from vlm_audit.attribution_rules import ATTRIBUTION_RULES, FINAL_STAGE_UPDATED, attribute_model
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 90)

TABLE_ORDER = list(ROSTER_TABLE_NAMES)
parameter_shapes = {name: load_parameter_shapes(name) for name in TABLE_ORDER}
print({name: len(shapes) for name, shapes in parameter_shapes.items()})

<home>\AppData\Local\Temp\claude\<session>\0edba254-4773-49bc-b009-31568158ee2c\scratchpad\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'CLIP-ViT-L/14': 590, 'SigLIP-L/16': 792, 'SigLIP2-So400m/14': 888, 'ViLT-B/32': 212, 'BLIP-large': 615, 'BLIP-2-OPT-2.7B': 1247, 'InstructBLIP-Vicuna-7B': 1096, 'LLaVA-1.5-7B': 686, 'LLaVA-NeXT-7B': 687, 'Molmo-7B-D': 585, 'IDEFICS-9B': 1015, 'Llama-3.2-11B-Vision': 906, 'Idefics2-8B': 763, 'Qwen2-VL-7B': 730, 'Qwen2.5-VL-7B': 729, 'InternVL3-8B': 685, 'Pixtral-12B': 585, 'DeepSeek-VL2-Small': 5639, 'Chameleon-7B': 548, 'Janus-Pro-7B': 935, 'Qwen3-VL-8B': 750, 'Gemma-4-12B': 629, 'Gemma-4-26B-A4B': 981}


---
## 1. One attribution rule table for every roster model

Every tensor is assigned by the first matching rule in `ATTRIBUTION_RULES[model]` (printed below). The conventions are stated once in `vlm_audit/attribution_rules.py`: **fusion** is everything that turns encoder output into language-side input (projectors, Q-Formers, resamplers, mergers, pooling connectors, learned layout vectors); **head** is every output projection to a vocabulary or answer space.

In [2]:
for name in ('Molmo-7B-D', 'ViLT-B/32', 'BLIP-2-OPT-2.7B', 'DeepSeek-VL2-Small'):
    print(name)
    for pattern, component in ATTRIBUTION_RULES[name]:
        print(f'    {pattern:<70} -> {component}')

Molmo-7B-D
    ^model\.vision_backbone\.image_vit\.                                   -> vision
    ^model\.vision_backbone\.                                              -> fusion
    ^model\.transformer\.ff_out\.                                          -> head
    ^model\.transformer\.                                                  -> language
ViLT-B/32
    ^vilt\.embeddings\.text_embeddings\.                                   -> language
    ^vilt\.embeddings\.token_type_embeddings                               -> fusion
    ^vilt\.embeddings\.                                                    -> vision
    ^vilt\.(encoder|layernorm)\.                                           -> language
    ^(vilt\.pooler|classifier)\.                                           -> head
BLIP-2-OPT-2.7B
    ^(query_tokens|qformer\.|language_projection\.)                        -> fusion
    ^vision_model\.                                                        -> vision
    ^language_model\.lm_he

In [3]:
def collapse(counts):
    """Aggregate rule components into the Table 2 columns (Janus generation modules into V/F/Head)."""
    vision = counts.get('vision', 0) + counts.get('vision_gen', 0)
    fusion = counts.get('fusion', 0) + counts.get('fusion_gen', 0)
    head = counts.get('head', 0) + counts.get('head_gen', 0)
    return (vision, fusion, counts.get('language', 0), head, counts.get('fused', 0), counts.get('vision_tokenizer', 0),
            counts.get('audio', 0))

attribution = {name: attribute_model(name, parameter_shapes[name]) for name in TABLE_ORDER}
rows = []
for name in TABLE_ORDER:
    result = attribution[name]
    vision, fusion, language, head, fused, tokenizer, audio = collapse(result['component_counts'])
    total = result['total']
    separable = fused == 0
    rows.append({'model': name, 'total_M': total / 1e6, 'V_M': vision / 1e6, 'F_M': fusion / 1e6, 'L_M': language / 1e6,
                 'Head_M': head / 1e6, 'fused_M': fused / 1e6, 'image_tokenizer_M': tokenizer / 1e6, 'audio_M': audio / 1e6,
                 'V_pct': 100 * vision / total if separable else None, 'F_pct': 100 * fusion / total if separable else None,
                 'L_pct': 100 * language / total if separable else None,
                 'V_over_L': vision / language if separable else None, 'residual': result['residual'],
                 'total_params': total, 'V_params': vision, 'F_params': fusion, 'L_params': language, 'Head_params': head})
table2_v2 = pd.DataFrame(rows).set_index('model')
assert (table2_v2['residual'] == 0).all(), 'every tensor must be attributed exactly once'
table2_v2.to_csv(MEASUREMENT_DIR / 'nb26_parameter_components_v2.csv')
table2_v2.sort_values('V_pct', ascending=False)[['total_M','V_M','F_M','L_M','Head_M','V_pct','F_pct','L_pct','V_over_L']].round(3)

,total_M,V_M,F_M,L_M,Head_M,V_pct,F_pct,L_pct,V_over_L
model,,,,,,,,,
CLIP-ViT-L/14,427.617,303.180,1.376,123.060,0.000,70.900,0.322,28.778,2.464
BLIP-large,446.290,303.691,0.000,141.977,0.623,68.048,0.000,31.813,2.139
SigLIP-L/16,652.478,316.284,0.000,336.195,0.000,48.474,0.000,51.526,0.941
SigLIP2-So400m/14,1136.008,428.226,0.000,707.783,0.000,37.696,0.000,62.304,0.605
BLIP-2-OPT-2.7B,3744.762,985.952,107.131,2651.679,0.000,26.329,2.861,70.810,0.372
InstructBLIP-Vicuna-7B,7913.727,985.952,188.835,6607.606,131.334,12.459,2.386,83.496,0.149
Llama-3.2-11B-Vision,10670.221,863.567,31.461,9249.856,525.337,8.093,0.295,86.689,0.093
Qwen2.5-VL-7B,8292.167,631.976,44.574,7070.619,544.997,7.621,0.538,85.269,0.089
Qwen2-VL-7B,8291.376,631.183,44.576,7070.619,544.997,7.613,0.538,85.277,0.089


### What changed against the A13 attribution (notebook 24), and why

Totals are identical (same tensors); only the column a tensor sits in moves.

In [4]:
A13_NAME = {'Llama-3.2-11B-Vision': 'Llama-3.2-11B-Vision-Instruct', 'Qwen2-VL-7B': 'Qwen2-VL-7B-Instruct',
            'Qwen2.5-VL-7B': 'Qwen2.5-VL-7B-Instruct'}
table2_a13 = pd.read_csv(MEASUREMENT_DIR / 'nb24_parameter_components_revised.csv').set_index('model')
REASONS = {
    'ViLT-B/32': 'A7: 23.47 M text word/position embeddings sat in V (the whole vilt.embeddings module); V keeps only the image patch projection, image CLS and image positions',
    'Molmo-7B-D': 'A8: the connector (2x2 attention pooling 7.34 M + SwiGLU projector 106.69 M) sat inside vision_backbone and was counted as V; LM head ff_out moved to Head',
    'BLIP-2-OPT-2.7B': 'convention: the language projection (Q-Former -> OPT) is fusion by the paper\'s own definition, not a head',
    'InstructBLIP-Vicuna-7B': 'convention: language projection to fusion; the untied Vicuna LM head to Head',
    'DeepSeek-VL2-Small': 'convention: untied LM head (209.7 M) to Head',
    'Janus-Pro-7B': 'convention: untied LM head (419.4 M) to Head (generation head already there)',
    'BLIP-large': 'convention: captioning LM-head transform to Head',
    'Chameleon-7B': 'no V/L decomposition; input embedding to the fused transformer, Head = LM head; image tokenizer reported separately',
}
change_rows = []
for name in TABLE_ORDER:
    if A13_NAME.get(name, name) not in table2_a13.index:
        continue   # added under decision D9 (2026-10-04): no A13 attribution to compare with
    before = table2_a13.loc[A13_NAME.get(name, name)]
    after = table2_v2.loc[name]
    deltas = {column: after[f'{column}_M'] - (before[f'{column}_M'] if pd.notna(before[f'{column}_M']) else 0.0) for column in ('V', 'F', 'L', 'Head')}
    if max(abs(value) for value in deltas.values()) > 0.05:
        change_rows.append({'model': name, **{f'd{column}_M': round(value, 2) for column, value in deltas.items()},
                            'V_pct_before': before['V_pct'], 'V_pct_after': round(after['V_pct'], 2) if pd.notna(after['V_pct']) else None,
                            'F_pct_before': before['F_pct'], 'F_pct_after': round(after['F_pct'], 2) if pd.notna(after['F_pct']) else None,
                            'reason': REASONS.get(name, '')})
changes = pd.DataFrame(change_rows).set_index('model')
changes.to_csv(MEASUREMENT_DIR / 'nb26_attribution_changes.csv')
changes

,dV_M,dF_M,dL_M,dHead_M,V_pct_before,V_pct_after,F_pct_before,F_pct_after,reason
model,,,,,,,,,
ViLT-B/32,-23.43,0.00,23.43,-0.02,22.1,2.10,0.0,0.00,A7: 23.47 M text word/position embeddings sat in V (the whole vilt.embeddings module);...
BLIP-large,-0.01,0.00,-0.62,0.62,68.0,68.05,0.0,0.00,convention: captioning LM-head transform to Head
BLIP-2-OPT-2.7B,-0.05,1.93,-0.02,-2.00,26.3,26.33,2.8,2.86,convention: the language projection (Q-Former -> OPT) is fusion by the paper's own def...
InstructBLIP-Vicuna-7B,-0.05,3.13,-131.29,128.23,12.5,12.46,2.3,2.39,convention: language projection to fusion; the untied Vicuna LM head to Head
Molmo-7B-D,-114.09,114.04,-545.02,545.00,5.0,3.63,0.0,1.42,A8: the connector (2x2 attention pooling 7.34 M + SwiGLU projector 106.69 M) sat insid...
DeepSeek-VL2-Small,0.03,0.04,-209.73,209.72,2.7,2.65,0.1,0.08,convention: untied LM head (209.7 M) to Head
Chameleon-7B,0.00,0.00,-6506.20,-268.46,NaN,NaN,NaN,NaN,"no V/L decomposition; input embedding to the fused transformer, Head = LM head; image ..."
Janus-Pro-7B,-0.03,0.03,-419.46,419.44,5.2,5.23,0.5,0.51,convention: untied LM head (419.4 M) to Head (generation head already there)


**Reading.** Two changes are substantive. **ViLT** falls from V% 22.1 to 2.1: the published V was the whole `vilt.embeddings` module, 90% of which is the *text* vocabulary. ViLT therefore is not the roster's most balanced design (Sec. 5.1) and leaves the intermediate band (Sec. 5.2). **Molmo** falls from V% 5.0 to 3.6 and gains F% 1.4, answering the reviewer's 'Molmo fusion = 0.0 M despite a connector'. The rest are head/fusion bookkeeping that make the Head column mean the same thing in every row.

---
## 2. Model identity (A8)

The commit measured is the latest commit at or before the measurement timestamp in each archived JSON record. For each, the weight and config blobs at that commit are compared with today's head.

In [5]:
from datetime import datetime, timezone
from huggingface_hub import HfApi
api = HfApi()
CHECKPOINT_KIND = {
    'CLIP-ViT-L/14': 'pre-trained (contrastive)', 'SigLIP-L/16': 'pre-trained (contrastive)', 'SigLIP2-So400m/14': 'pre-trained (contrastive)',
    'ViLT-B/32': 'fine-tuned on VQAv2', 'BLIP-large': 'fine-tuned on COCO captioning', 'BLIP-2-OPT-2.7B': 'pre-trained only (stage 2)',
    'InstructBLIP-Vicuna-7B': 'instruction-tuned', 'LLaVA-1.5-7B': 'instruction-tuned', 'LLaVA-NeXT-7B': 'instruction-tuned (Mistral-7B-Instruct-v0.2 LLM)',
    'Molmo-7B-D': 'instruction-tuned (SFT)', 'IDEFICS-9B': 'BASE (not the -instruct release)', 'Llama-3.2-11B-Vision': 'instruction-tuned (-Instruct)',
    'Idefics2-8B': 'instruction-tuned (idefics2-8b, not -base)', 'Qwen2-VL-7B': 'instruction-tuned (-Instruct)', 'Qwen2.5-VL-7B': 'instruction-tuned (-Instruct)',
    'InternVL3-8B': 'MPO-tuned release (on InternVL3-8B-Instruct)', 'Pixtral-12B': 'instruction-tuned (HF port of mistralai/Pixtral-12B-2409)',
    'DeepSeek-VL2-Small': 'SFT release', 'Chameleon-7B': 'safety-tuned release', 'Janus-Pro-7B': 'SFT release (stage III)',
    'Qwen3-VL-8B': 'instruction-tuned (-Instruct; SFT, distillation, RL)', 'Gemma-4-12B': 'instruction-tuned (-it)',
    'Gemma-4-26B-A4B': 'instruction-tuned (-it)',
}

def blob_ids(repository, revision):
    """Weight and config blob hashes of one revision."""
    information = api.model_info(repository, revision=revision, files_metadata=True)
    return {sibling.rfilename: (sibling.lfs.sha256 if sibling.lfs else sibling.blob_id) for sibling in information.siblings
            if sibling.rfilename.endswith(('.safetensors', '.bin', 'config.json'))}

D9_MEASURED_ON = datetime(2026, 10, 4).date()
identity_rows = []
for name in TABLE_ORDER:
    roster_name, repository = ROSTER_TABLE_NAMES[name]
    archive_path = ARCHIVE_DIR / f"{repository.replace('/', '_')}.json"
    if not archive_path.exists():
        # Added under decision D9 (2026-10-04): measured at the head commit read that day.
        information = api.model_info(repository)
        identity_rows.append({'model': name, 'repository': repository, 'checkpoint': CHECKPOINT_KIND[name],
                              'measured_on': D9_MEASURED_ON, 'commit_measured': information.sha[:12],
                              'commit_date': information.last_modified.date(), 'head_today': information.sha[:12],
                              'weights_or_config_changed_since': 'none', 'how_established': 'added under D9: head commit',
                              'config_read_from': CONFIG_MIRRORS.get(repository, repository)})
        continue
    archived = json.loads(archive_path.read_text(encoding='utf-8'))
    measured_at = datetime.fromisoformat(archived['measurement_timestamp']).replace(tzinfo=timezone.utc)
    try:
        commits = api.list_repo_commits(repository)
        measured_commit = next(commit for commit in commits if commit.created_at <= measured_at)
        head_commit = commits[0]
        changed = [] if measured_commit.commit_id == head_commit.commit_id else sorted(
            file for file in set(blob_ids(repository, measured_commit.commit_id)) | set(blob_ids(repository, head_commit.commit_id))
            if blob_ids(repository, measured_commit.commit_id).get(file) != blob_ids(repository, head_commit.commit_id).get(file))
        measured_sha, measured_date, head_sha = measured_commit.commit_id, measured_commit.created_at.date(), head_commit.commit_id
        how = 'commit history'
    except Exception:
        # Gated repositories: the commit list needs a licence token, but the head sha and its date are public.
        information = api.model_info(repository)
        assert information.last_modified <= measured_at, 'gated repo changed after measurement'
        measured_sha, measured_date, head_sha, changed = information.sha, information.last_modified.date(), information.sha, []
        how = 'gated: head unchanged since ' + str(information.last_modified.date())
    identity_rows.append({'model': name, 'repository': repository, 'checkpoint': CHECKPOINT_KIND[name],
                          'measured_on': measured_at.date(), 'commit_measured': measured_sha[:12], 'commit_date': measured_date,
                          'head_today': head_sha[:12], 'weights_or_config_changed_since': ', '.join(changed) or 'none',
                          'how_established': how, 'config_read_from': CONFIG_MIRRORS.get(repository, repository)})
identity = pd.DataFrame(identity_rows).set_index('model')
identity.to_csv(MEASUREMENT_DIR / 'nb26_model_identity.csv')
identity

,repository,checkpoint,measured_on,commit_measured,commit_date,head_today,weights_or_config_changed_since,how_established,config_read_from
model,,,,,,,,,
CLIP-ViT-L/14,openai/clip-vit-large-patch14,pre-trained (contrastive),2026-05-21,32bd64288804,2023-09-15,32bd64288804,none,commit history,openai/clip-vit-large-patch14
SigLIP-L/16,google/siglip-large-patch16-384,pre-trained (contrastive),2026-05-21,ce005573a409,2024-09-26,ce005573a409,none,commit history,google/siglip-large-patch16-384
SigLIP2-So400m/14,google/siglip2-so400m-patch14-384,pre-trained (contrastive),2026-05-21,e8e487298228,2025-02-21,e8e487298228,none,commit history,google/siglip2-so400m-patch14-384
ViLT-B/32,dandelin/vilt-b32-finetuned-vqa,fine-tuned on VQAv2,2026-05-21,d0a1f6ab8852,2022-08-02,d0a1f6ab8852,none,commit history,dandelin/vilt-b32-finetuned-vqa
BLIP-large,Salesforce/blip-image-captioning-large,fine-tuned on COCO captioning,2026-05-21,353689b859fc,2025-02-03,353689b859fc,none,commit history,Salesforce/blip-image-captioning-large
BLIP-2-OPT-2.7B,Salesforce/blip2-opt-2.7b,pre-trained only (stage 2),2026-05-21,59a1ef6c1e51,2025-02-03,59a1ef6c1e51,none,commit history,Salesforce/blip2-opt-2.7b
InstructBLIP-Vicuna-7B,Salesforce/instructblip-vicuna-7b,instruction-tuned,2026-05-21,19103d0c5b52,2025-02-03,19103d0c5b52,none,commit history,Salesforce/instructblip-vicuna-7b
LLaVA-1.5-7B,llava-hf/llava-1.5-7b-hf,instruction-tuned,2026-05-21,b234b804b114,2025-06-06,b234b804b114,none,commit history,llava-hf/llava-1.5-7b-hf
LLaVA-NeXT-7B,llava-hf/llava-v1.6-mistral-7b-hf,instruction-tuned (Mistral-7B-Instruct-v0.2 LLM),2026-05-21,2424fdd47412,2025-12-22,2424fdd47412,none,commit history,llava-hf/llava-v1.6-mistral-7b-hf


In [6]:
# LLaVA-NeXT: which language model does the measured checkpoint contain?
from transformers import AutoConfig
llava_next = AutoConfig.from_pretrained('llava-hf/llava-v1.6-mistral-7b-hf')
print('text_config.model_type:', llava_next.text_config.model_type, '| vocab', llava_next.text_config.vocab_size,
      '| kv heads', llava_next.text_config.num_key_value_heads, '| intermediate', llava_next.text_config.intermediate_size)
print('Language parameters (Table 2 L):', f"{table2_v2.loc['LLaVA-NeXT-7B', 'L_M']:,.1f} M", '-> Mistral-7B, not Vicuna-1.5-7B (', 
      f"{table2_v2.loc['LLaVA-1.5-7B', 'L_M']:,.1f} M )")

text_config.model_type: mistral | vocab 32064 | kv heads 8 | intermediate 14336
Language parameters (Table 2 L): 7,110.9 M -> Mistral-7B, not Vicuna-1.5-7B ( 6,607.6 M )


**Reading.** All 20 commits measured in May 2026 are identified (the three models added under D9 are recorded at their head commit of 2026-10-04); none has changed weights or config since the May 2026 runs. The reviewer is right that the LLaVA-NeXT checkpoint is the **Mistral-7B** variant (grouped-query attention, 14,336-wide MLP), so the LLaVA-1.5 → LLaVA-NeXT comparison also changes the language-model family. IDEFICS-9B was measured as the **base** release while all other generative models are instruction-tuned; this matters for the trainable share (Section 3) and for benchmark comparability.

---
## 3. Final-stage trainable shares from primary sources (A3)

`FINAL_STAGE_UPDATED` lists, per model, the tensors updated in the final multimodal training stage of the measured checkpoint, the confidence level and the source. Quotes and section numbers are in `paper/Revision_Decision_Log.md`.

In [7]:
PUBLISHED_TABLE5 = {'CLIP-ViT-L/14': '100.0', 'ViLT-B/32': '100.0', 'BLIP-large': '100.0', 'SigLIP-L/16': '100.0',
                    'BLIP-2-OPT-2.7B': '2.8', 'InstructBLIP-Vicuna-7B': '2.3', 'IDEFICS-9B': '~9-19% (base) / 100% (instruct)',
                    'LLaVA-1.5-7B': '95.6', 'LLaVA-NeXT-7B': '95.9', 'Molmo-7B-D': '95.0', 'Llama-3.2-11B-Vision': '~12-20%',
                    'Idefics2-8B': '95', 'Qwen2-VL-7B': '100.0', 'Pixtral-12B': '100.0', 'DeepSeek-VL2-Small': '97.3',
                    'Chameleon-7B': '100.0', 'SigLIP2-So400m/14': '100.0', 'Qwen2.5-VL-7B': '100.0', 'InternVL3-8B': '100.0',
                    'Janus-Pro-7B': '99.1'}
trainable_rows = []
for name in TABLE_ORDER:
    patterns, confidence, source = FINAL_STAGE_UPDATED[name]
    trainable = attribution[name]['trainable']
    share = round(100 * trainable / attribution[name]['total'], 2) if trainable is not None else float('nan')
    trainable_rows.append({'model': name, 'published': PUBLISHED_TABLE5.get(name, 'not in submitted paper'),
                           'revised_pct': share,
                           'confidence': confidence, 'source': source})
table5_v2 = pd.DataFrame(trainable_rows).set_index('model')
table5_v2.to_csv(MEASUREMENT_DIR / 'nb26_trainable_share_v2.csv')
table5_v2.sort_values('revised_pct')

,published,revised_pct,confidence,source
model,,,,
InstructBLIP-Vicuna-7B,2.3,2.39,verified,Dai et al. 2023 Sec. 2.6; LAVIS keeps llm_proj trainable
BLIP-2-OPT-2.7B,2.8,2.86,verified,Li et al. 2023 Sec. 3.3; LAVIS freezes ViT and OPT; ln_vision trained
IDEFICS-9B,~9-19% (base) / 100% (instruct),17.48,verified,"idefics-9b model card: backbones frozen, new parameters trained (base checkpoint)"
Llama-3.2-11B-Vision,~12-20%,24.74,verified (SFT); DPO names only 'vision adapters',"Llama 3 paper Secs. 7.2, 7.4, 7.5.2; Meta Llama 3.2 blog: LLM never updated"
Qwen2.5-VL-7B,100.0,92.38,verified (ViT frozen); merger source silent,Qwen2.5-VL report Sec. 2.3.4: SFT and DPO with the ViT frozen
Qwen2-VL-7B,100.0,92.39,verified (ViT locked); merger from released weights,Wang et al. 2024 Sec. 2.2: 'we lock the ViT parameters' in the final stage
LLaVA-1.5-7B,95.6,95.70,verified,Liu et al. 2023 (LLaVA) Sec. 4.2; vision tower identical across 7B/13B releases
Gemma-4-26B-A4B,not in submitted paper,97.79,partially verified (report statement; vision encoder bit-identical pre-trained vs inst...,Gemma 4 report Sec. 1: 'Gemma 4 models have frozen vision and audio encoders'
Janus-Pro-7B,99.1,99.03,partially verified,"Janus Sec. 3.2 Stage III, restated in Janus-Pro Sec. 2.2: all but the generation encoder"


**Reading.** The published Table 5 misstates eight rows. Three models trained their vision encoder in the final stage although Table 5 lists it as frozen (**LLaVA-NeXT**, **Molmo**, **DeepSeek-VL2**: 100%). Two froze it although Table 5 lists nothing frozen (**Qwen2-VL**, **Qwen2.5-VL**: 92.4%). The cross-attention models are 17.5% (IDEFICS-9B base) and 24.7% (Llama-3.2-Vision). **Idefics2** adapts both backbones with merged LoRA/DoRA, so every tensor changed, but at low rank: 100% updated, 8.8% fully trained — a choice of definition the manuscript must state. The distribution is not bimodal: it has a bridge-only cluster (2.4–2.9%), a cross-attention cluster (17–25%), a frozen-vision cluster (92–96%) and a fully trained majority (99–100%).